In [2]:
import sys
import os
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

# Setup paths
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.explainers.lime_text import LimeTextExplainer # Assumes this exists from previous steps
from src.core.sp_lime import SubmodularPick

def run_experiment():
    print("--- Reproducing Section 6.2: Trusting the Model (Text) ---")
    
    # 1. Load Data
    # We use Atheism vs Christian groups. 
    # This dataset is notorious because headers (which should be removed) contain dead giveaways.
    categories = ['alt.atheism', 'soc.religion.christian']
    print("Loading 20 Newsgroups data (Train/Test)...")
    newsgroups_train = fetch_20newsgroups(subset='train', categories=categories)
    newsgroups_test = fetch_20newsgroups(subset='test', categories=categories)
    
    # 2. Train an "Untrustworthy" Classifier
    # We purposefully do NOT remove headers. The model will learn that emails from 
    # specific servers (e.g., 'cl.cam.ac.uk') are always Christian.
    print("Training SVM on raw text (including artifacts)...")
    vectorizer = TfidfVectorizer(lowercase=False) 
    clf = SVC(probability=True, kernel='linear') # SVM is what the paper used
    pipeline = make_pipeline(vectorizer, clf)
    pipeline.fit(newsgroups_train.data, newsgroups_train.target)
    
    acc = pipeline.score(newsgroups_test.data, newsgroups_test.target)
    print(f"Test Accuracy: {acc:.2f} (Looks great, right?)")
    
    # 3. Generate LIME Explanations for a Batch
    # In real life, you'd run this on your validation set.
    # We run on 30 random instances to simulate a batch.
    N_SAMPLES = 30
    print(f"\nGenerating LIME explanations for {N_SAMPLES} random instances...")
    
    indices = np.random.choice(len(newsgroups_test.data), N_SAMPLES, replace=False)
    text_data = [newsgroups_test.data[i] for i in indices]
    
    explainer = LimeTextExplainer(random_state=42)
    explanations = []
    
    for i, text in enumerate(text_data):
        # Predict
        probs = pipeline.predict_proba([text])[0]
        pred_class = np.argmax(probs)
        
        # Explain the PREDICTED class
        exp = explainer.explain_instance(
            text, 
            pipeline.predict_proba, 
            labels=(pred_class,), 
            num_features=5,
            num_samples=200 # Lower samples for speed in this demo
        )
        explanations.append(exp[pred_class])
        
        sys.stdout.write(f"\rExplained {i+1}/{N_SAMPLES}")
        sys.stdout.flush()
        
    print("\n\n--- Running Submodular Pick (Budget = 4) ---")
    # We want to pick 4 instances that cover the most important features
    sp = SubmodularPick(explanations, num_features_to_select=5)
    selected_indices = sp.pick_instances(budget_B=4)
    
    print(f"SP-LIME selected instances indices: {selected_indices}")
    
    print("\n--- RESULTS: Why you should NOT trust this model ---")
    print("SP-LIME should surface at least one instance where the model relies on HEADERS (Cheating).")
    
    for i, idx in enumerate(selected_indices):
        print(f"\n[Selected Instance {i+1}]")
        exp = explanations[idx]
        print(f"Predicted Class: {newsgroups_train.target_names[exp['target_class']]}")
        print("Top Features Used:")
        
        # Check if it found artifacts
        found_artifact = False
        for feature, weight in exp['explanation_map']:
            # Highlight artifacts commonly found in this dataset
            is_artifact = any(x in feature.lower() for x in ['host', 'nntp', 'edu', 'organization', 'path', '@'])
            marker = "  <-- SUSPICIOUS (Data Leakage)" if is_artifact else ""
            print(f"  {feature:>20} : {weight:.4f} {marker}")
            if is_artifact: found_artifact = True
            
        if found_artifact:
            print("  -> ANALYSIS: The model is reading email headers/metadata, not the content!")



In [3]:

run_experiment()

--- Reproducing Section 6.2: Trusting the Model (Text) ---
Loading 20 Newsgroups data (Train/Test)...
Training SVM on raw text (including artifacts)...
Test Accuracy: 0.95 (Looks great, right?)

Generating LIME explanations for 30 random instances...
Explained 30/30

--- Running Submodular Pick (Budget = 4) ---
SP-LIME selected instances indices: [22, 1, 27, 20]

--- RESULTS: Why you should NOT trust this model ---
SP-LIME should surface at least one instance where the model relies on HEADERS (Cheating).

[Selected Instance 1]
Predicted Class: alt.atheism
Top Features Used:
                   TIN : 0.1792 
                    No : 0.1231 
                   has : 0.0840 
               Pihatie : 0.0823 
                friend : 0.0494 

[Selected Instance 2]
Predicted Class: alt.atheism
Top Features Used:
                 still : 0.1200 
                simply : 0.1114 
          Organization : 0.0720   <-- SUSPICIOUS (Data Leakage)
                   all : 0.0711 
            convinci